<h1>DWH<h1>

Imports

In [1]:
import pandas as pd
import numpy as np
import sqlite3

from datetime import date

TODAY      = date.today().isoformat()   # bijv. '2025-05-02'
FAR_PAST = '1900-01-01'
END_TIME   = '9999-12-31'

Logging creëeren

In [2]:
import logging
import os
os.makedirs("logs", exist_ok=True)

logging.basicConfig( 
    filename="logs/dwh_de.log",
    level=logging.INFO, 
    format="%(asctime)s | %(name)s |  %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H-%M-%S",
    force=True
    )

logger = logging.getLogger("DWH")
logger.setLevel(logging.INFO)

Data connecties

In [3]:
#sdm connectie
sdm_conn = sqlite3.connect("Data/sdm-dwh/sdmdb.db")
sdm_cursor = sdm_conn.cursor()

#dwh connectie
dwh_conn = sqlite3.connect("Data/sdm-dwh/dwhdb.db")
dwh_cursor = dwh_conn.cursor()

DWH creëeren

In [4]:
dwh_sql_script = """
CREATE TABLE IF NOT EXISTS DIM_age_group (
    AGE_GROUP_CODE INT PRIMARY KEY,
    UPPER_AGE INT,
    LOWER_AGE INT
);

CREATE TABLE IF NOT EXISTS DIM_customer_segment (
    SEGMENT_CODE INT PRIMARY KEY,
    LANGUAGE VARCHAR(10),
    SEGMENT_NAME VARCHAR(45),
    SEGMENT_DESCRIPTION VARCHAR(100)
);

CREATE TABLE IF NOT EXISTS DIM_country (
    COUNTRY_CODE INT PRIMARY KEY,
    COUNTRY VARCHAR(45),
    LANGUAGE VARCHAR(10),
    CURRENCY VARCHAR(45),
    FLAG_IMAGE VARCHAR(100),
    TERRITORY_NAME_EN VARCHAR(45)
);

CREATE TABLE IF NOT EXISTS DIM_customer_headquarters (
    CUSTOMER_CODE_HQ INT PRIMARY KEY,
    CUSTOMER_NAME VARCHAR(100),
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(45),
    REGION VARCHAR(45),
    POSTAL_ZONE VARCHAR(45),
    PHONE VARCHAR(45),
    FAX VARCHAR(45),
    COUNTRY INT,
    SEGMENT_CODE INT,
    FOREIGN KEY (COUNTRY) REFERENCES DIM_country(COUNTRY_CODE),
    FOREIGN KEY (SEGMENT_CODE) REFERENCES DIM_customer_segment(SEGMENT_CODE)
);

CREATE TABLE IF NOT EXISTS DIM_customer (
    CUSTOMER_CODE INT PRIMARY KEY,
    COMPANY_NAME VARCHAR(100),
    CUSTOMER_TYPE_EN VARCHAR(100),
    CUSTOMER_HEADQUARTERS INT,
    FOREIGN KEY (CUSTOMER_HEADQUARTERS) REFERENCES DIM_customer_headquarters(CUSTOMER_CODE_HQ)
);

CREATE TABLE IF NOT EXISTS FACT_sales_demographic(
    DEMOGRAPHIC_CODE INT PRIMARY KEY,
    SALES_PERCENT INT,
    CUSTOMER_HEADQUARTERS INT,
    AGE_GROUP INT,
    FOREIGN KEY (CUSTOMER_HEADQUARTERS) REFERENCES DIM_customer_headquarters(CUSTOMER_CODE_HQ),
    FOREIGN KEY (AGE_GROUP) REFERENCES DIM_age_group(AGE_GROUP_CODE)
);

--SCD Type 2
CREATE TABLE IF NOT EXISTS DIM_product(
    PRODUCT_NUMBER INT PRIMARY KEY,
    DESCRIPTION VARCHAR(100),
    INTRODUCTION_DATE DATE,
    COST_CATEGORY VARCHAR(45),
    MARGIN DECIMAL(10,2),
    PRODUCT_IMG VARCHAR(45),
    LANGUAGE VARCHAR(10),
    PRODUCT_NAME VARCHAR(100),
    PRODUCT_TYPE_EN VARCHAR(100),
    PRODUCT_LINE_EN VARCHAR(100),
    START_DATE VARCHAR(100),
    END_DATE VARCHAR(100),
    ACTIVE_INDICATOR BOOLEAN
);

--SCD Type 2
CREATE TABLE IF NOT EXISTS DIM_retailer_site(
    SK_RETAILER_SITE_CODE INT PRIMARY KEY,
    RETAILER_SITE_CODE INT,
    RETAILER_CODE INT,
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(100),
    REGION VARCHAR(100),
    POSTAL_CODE VARCHAR(45),
    COUNTRY INT,
    CUSTOMER INT,
    START_DATE VARCHAR(100),
    END_DATE VARCHAR(100),
    ACTIVE_INDICATOR BOOLEAN,
    ORIGIN VARCHAR(100),
    FOREIGN KEY (CUSTOMER) REFERENCES DIM_customer(CUSTOMER_CODE),
    FOREIGN KEY (COUNTRY) REFERENCES DIM_country(COUNTRY_CODE)
);

CREATE TABLE IF NOT EXISTS DIM_sales_branch(
    SALES_BRANCH_CODE INT PRIMARY KEY,
    ADRESS1 VARCHAR(100),
    ADRESS2 VARCHAR(100),
    CITY VARCHAR(100),
    REGION VARCHAR(100),
    POSTAL_CODE VARCHAR(45),
    COUNTRY INT,
    FOREIGN KEY (COUNTRY) REFERENCES DIM_country(COUNTRY_CODE)
);

CREATE TABLE IF NOT EXISTS DIM_sales_staff(
    SALES_STAFF_CODE INT PRIMARY KEY,
    FIRST_NAME VARCHAR(45),
    LAST_NAME VARCHAR(45),
    POSITION_EN VARCHAR(100),
    WORK_PHONE VARCHAR(45),
    EXTENSION INT,
    FAX VARCHAR(45),
    DATE_HIRED DATE,
    TRAINING_YEAR INT,
    TRAINING_COURSE VARCHAR(100),
    STATISFACTION_YEAR INT,
    STATISFACTION_DESCRIPTION VARCHAR(100),
    SALES_BRANCH INT,
    MANAGER INT,
    FOREIGN KEY (SALES_BRANCH) REFERENCES DIM_sales_branch(SALES_BRANCH_CODE),
    FOREIGN KEY (MANAGER) REFERENCES DIM_sales_staff(SALES_STAFF_CODE)
);

CREATE TABLE IF NOT EXISTS FACT_order_header(
    ORDER_NUMBER INT PRIMARY KEY,
    RETAILER_NAME VARCHAR(100),
    RETAILER_CONTACT_CODE INT,
    ORDER_DATE DATE,
    ORDER_METHOD_EN VARCHAR(100),
    RETAILER_SITE INT,
    SALES_STAFF INT,
    SALES_BRANCH INT,
    FOREIGN KEY (RETAILER_SITE) REFERENCES DIM_retailer_site(RETAILER_SITE_CODE),
    FOREIGN KEY (SALES_STAFF) REFERENCES DIM_sales_staff(SALES_STAFF_CODE),
    FOREIGN KEY (SALES_BRANCH) REFERENCES DIM_sales_branch(SALES_BRANCH_CODE)
);

CREATE TABLE IF NOT EXISTS FACT_order_details(
    ORDER_DETAIL_CODE INT PRIMARY KEY,
    QUANTITY INT,
    UNIT_COST DECIMAL(10,2),
    UNIT_REVENUE DECIMAL(10,2),
    UNIT_PRICE DECIMAL(10,2),
    UNIT_SALE_PRICE DECIMAL(10,2),
    ORDER_HEADER INT,
    PRODUCT INT,
    FOREIGN KEY (ORDER_HEADER) REFERENCES FACT_order_header(ORDER_NUMBER),
    FOREIGN KEY (PRODUCT) REFERENCES DIM_product(PRODUCT_NUMBER)
);

CREATE TABLE IF NOT EXISTS DIM_returned_item(
    RETURN_CODE INT PRIMARY KEY,
    RETURN_DATE DATE,
    QUANTITY_CATEGORY VARCHAR(100),
    RETURN_REASON_EN VARCHAR(100),
    ORDER_DETAILS INT,
    FOREIGN KEY (ORDER_DETAILS) REFERENCES FACT_order_details(ORDER_DETAIL_CODE)
);

CREATE TABLE IF NOT EXISTS DIM_inventory_level(
    INVENTORY_LEVELS_CODE INT PRIMARY KEY,
    INVENTORY_YEAR INT,
    INVENTORY_MONTH INT,
    INVENTORY_COUNT INT,
    PRODUCT INT,
    FOREIGN KEY(PRODUCT) REFERENCES DIM_product(PRODUCT_NUMBER)
);

CREATE TABLE IF NOT EXISTS DIM_product_forecast(
    PRODUCT_FORECAST_CODE INT PRIMARY KEY,
    YEAR INT,
    MONTH INT,
    EXPECTED_VOLUME INT,
    PRODUCT INT,
    FOREIGN KEY(PRODUCT) REFERENCES DIM_product(PRODUCT_NUMBER)
);

CREATE TABLE IF NOT EXISTS FACT_sales_target(
    SALES_TARGET_CODE INT PRIMARY KEY,
    SALES_YEAR INT,
    SALES_PERIOD INT,
    RETAILER_NAME VARCHAR(100),
    RETAILER_CODE INT,
    SALES_TARGET DECIMAL(10,2),
    SALES_STAFF INT,
    PRODUCT INT,
    FOREIGN KEY (SALES_STAFF) REFERENCES DIM_sales_staff(SALES_STAFF_CODE),
    FOREIGN KEY (PRODUCT) REFERENCES DIM_product(PRODUCT_NUMBER)
);

CREATE TABLE IF NOT EXISTS DIM_Month(
    MONTH_ID INT PRIMARY KEY,
    MONTHNR INT,
    MONTH VARCHAR(45),
    QUARTER INT,
    YEAR INT
);

CREATE TABLE IF NOT EXISTS DIM_Date(
    DATE_ID INT PRIMARY KEY,
    DATUM DATE,
    MONTH INT,
    FOREIGN KEY (MONTH) REFERENCES DIM_Month(MONTH_ID)
);
"""

dwh_cursor.executescript(dwh_sql_script)
dwh_conn.commit()

SDM uitlezen

In [5]:
age_group = pd.read_sql("SELECT * FROM age_group;",sdm_conn)
customer_contact = pd.read_sql("SELECT * FROM customer_contact;",sdm_conn)
customer_segment = pd.read_sql("SELECT * FROM customer_segment;",sdm_conn)
customer_type = pd.read_sql("SELECT * FROM customer_type;",sdm_conn)
sales_territory = pd.read_sql("SELECT * FROM sales_territory;",sdm_conn)
customer_store = pd.read_sql("SELECT * FROM customer_store;",sdm_conn)
crm_country = pd.read_sql("SELECT * FROM crm_country;",sdm_conn)
customer = pd.read_sql("SELECT * FROM customer;",sdm_conn)
sales_demographic = pd.read_sql("SELECT * FROM sales_demographic;",sdm_conn)
customer_headquarters = pd.read_sql("SELECT * FROM customer_headquarters;",sdm_conn)

order_method = pd.read_sql("SELECT * FROM order_method;",sdm_conn)
product_line = pd.read_sql("SELECT * FROM product_line;",sdm_conn)
retailer_site = pd.read_sql("SELECT * FROM retailer_site;",sdm_conn)
return_reason = pd.read_sql("SELECT * FROM return_reason;",sdm_conn)
returned_item = pd.read_sql("SELECT * FROM returned_item;",sdm_conn)
order_details = pd.read_sql("SELECT * FROM order_details;",sdm_conn)
order_header = pd.read_sql("SELECT * FROM order_header;",sdm_conn)
product_type = pd.read_sql("SELECT * FROM product_type;",sdm_conn)
product = pd.read_sql("SELECT * FROM product;",sdm_conn)
sales_staff = pd.read_sql("SELECT * FROM sales_staff;",sdm_conn)
go_sales_country = pd.read_sql("SELECT * FROM go_sales_country;",sdm_conn)
sales_branch = pd.read_sql("SELECT * FROM sales_branch;",sdm_conn)

satisfaction = pd.read_sql("SELECT * FROM satisfaction;",sdm_conn)
training = pd.read_sql("SELECT * FROM training;",sdm_conn)
course = pd.read_sql("SELECT * FROM course;",sdm_conn)
sales_office = pd.read_sql("SELECT * FROM sales_office;",sdm_conn)
satisfaction_type = pd.read_sql("SELECT * FROM satisfaction_type;",sdm_conn)
sales_representative = pd.read_sql("SELECT * FROM sales_representative;",sdm_conn)

inventory_levels = pd.read_sql("SELECT * FROM inventory_level", sdm_conn)
product_forecast = pd.read_sql("SELECT * FROM product_forecast", sdm_conn)
sales_target = pd.read_sql("SELECT * FROM sales_target", sdm_conn)

Lijst DWH tabellen

In [6]:
alle_dwh_tabellen = {
    'Data/dwhdb.db': {
    "DIM_Date" : 'DIM_Date',
    "DIM_Month" : 'DIM_Month',
    "DIM_age_group" : 'DIM_age_group',
    "DIM_country" : 'DIM_country',
    "DIM_customer" : 'DIM_customer',
    "DIM_customer_headquarters" : 'DIM_customer_headquarters',
    "DIM_inventory_level" : 'DIM_inventory_level',
    "DIM_product" : 'DIM_product',
    "DIM_product_forecast" : 'DIM_product_forecast',
    "DIM_retailer_site" : 'DIM_retailer_site',
    "DIM_returned_item" : 'DIM_returned_item',
    "DIM_sales_branch" : 'DIM_sales_branch',
    "DIM_sales_staff" : 'DIM_sales_staff',
    "FACT_order_details" : 'FACT_order_details',
    "FACT_order_header" : 'FACT_order_header',
    "FACT_sales_demographic" : 'FACT_sales_demographic',
    "FACT_sales_target" : 'FACT_sales_target'
    }
 }

# for db, tables in alle_dwh_tabellen.items():
#     for tabel in tables:
#         print(tabel)

DIM_sales_staff

In [7]:
sales_representative = sales_representative.drop(columns='EXTENSION')

DIM_sales_Staff = pd.merge(
    sales_representative,
    sales_staff,
    how='outer',
    on=['FIRST_NAME', 'LAST_NAME', 'POSITION_EN',
       'WORK_PHONE', 'FAX', 'DATE_HIRED']
)

DIM_sales_Staff.insert(0, 'SALES_STAFF_CODE', DIM_sales_Staff.pop('SALES_STAFF_CODE'))
DIM_sales_Staff['DATE_HIRED'] = pd.to_datetime(DIM_sales_Staff['DATE_HIRED'], format="%d-%b-%Y %I:%M:%S %p")

training = training.rename(columns={'SALES_REPRESENTATIVE' : 'SALES_REPRESENTATIVE_CODE'})
satisfaction = satisfaction.rename(columns={'SALES_REPRESENTATIVE' : 'SALES_REPRESENTATIVE_CODE'})
satisfaction_type = satisfaction_type.rename(columns={'SATISFACTION_TYPE_CODE' : 'SATISFACTION_TYPE'})

DIM_sales_Staff = DIM_sales_Staff.merge(training, how='left', on='SALES_REPRESENTATIVE_CODE')
DIM_sales_Staff = DIM_sales_Staff.rename(columns={'MANAGER_CODE':'MANAGER', 'YEAR': 'TRAINING_YEAR', 'TRAINING_CODE':'TRAINING_COURSE'})
DIM_sales_Staff = DIM_sales_Staff.merge(satisfaction, how='left', on='SALES_REPRESENTATIVE_CODE')
DIM_sales_Staff = DIM_sales_Staff.merge(satisfaction_type, how='left', on='SATISFACTION_TYPE')
DIM_sales_Staff = DIM_sales_Staff.rename(columns={'YEAR': 'STATISFACTION_YEAR', 'SATISFACTION_TYPE_DESCRIPTION': 'STATISFACTION_DESCRIPTION'})
DIM_sales_Staff['DATE_HIRED'] = DIM_sales_Staff['DATE_HIRED'].astype(str)


#DIM_sales_Staff

DIM_country

In [8]:
crm_country = crm_country.rename(columns={'COUNTRY_EN':'COUNTRY'})

DIM_Country = pd.merge(
    go_sales_country,
    crm_country,
    how='outer',
    on=['COUNTRY_CODE', 'COUNTRY']
)

DIM_Country = DIM_Country.rename(columns={'SALES_TERRITORY':'SALES_TERRITORY_CODE'})
DIM_Country = pd.merge(
    DIM_Country,
    sales_territory,
    how='left',
    on=['SALES_TERRITORY_CODE']
)

DIM_Country = DIM_Country.drop(columns='SALES_TERRITORY_CODE')
DIM_Country = DIM_Country.rename(columns={'CURRENCY_NAME' : 'CURRENCY'})

DIM_Country

,COUNTRY_CODE,COUNTRY,LANGUAGE,CURRENCY,FLAG_IMAGE,TERRITORY_NAME_EN
0,1,France,EN,francs,F01,Central Europe
1,2,Germany,EN,marks,F02,Central Europe
2,3,United States,EN,dollars,F03,Americas
3,4,Canada,EN,dollars,F04,Americas
4,5,Austria,EN,schillings,F05,Southern Europe
5,6,Italy,EN,lira,F06,Southern Europe
6,7,Netherlands,EN,guilders,NaN,NaN
7,8,Switzerland,EN,francs,F08,Central Europe
8,9,United Kingdom,EN,pounds,F09,Central Europe
9,10,Sweden,EN,krona,F10,Northern Europe


DIM_sales_branch

In [9]:
sales_office = sales_office.rename(columns={'STREET' : 'ADRESS1', 'ADDITION' : 'ADRESS2','ZIPCODE':'POSTAL_CODE'})

DIM_Sales_Branch = pd.merge(
    sales_branch,
    sales_office,
    how='outer',
    on=['ADRESS1','ADRESS2','CITY', 'REGION', 'POSTAL_CODE','COUNTRY']
)

#DIM_Sales_Branch

DIM_Retailer_Site

In [10]:
customer_store = customer_store.rename(columns={'CUSTOMER_STORE_CODE':'RETAILER_SITE_CODE',
                                                'ZIPCODE':'POSTAL_ZONE',
                                                'STATE': 'REGION',
                                                'CRM_COUNTRY' : 'COUNTRY',
                                                'STREET' : 'ADRESS1',
                                                'ADDITION' : 'ADRESS2'})

customer_store["ORIGIN"] = "customer_store"
retailer_site["ORIGIN"] = "retailer_site"

DIM_Retailer_Site = pd.merge(
    retailer_site,
    customer_store,
    how='outer',
    on=['RETAILER_SITE_CODE', 'ADRESS1', 'ADRESS2', 'CITY', 'REGION', 'POSTAL_ZONE', 'COUNTRY', 'ACTIVE_INDICATOR', 'ORIGIN']
)

DIM_Retailer_Site = DIM_Retailer_Site.rename(columns={'POSTAL_ZONE' : 'POSTAL_CODE'})
DIM_Retailer_Site["START_DATE"] = pd.Timestamp.now().strftime("%Y-%m-%d")
DIM_Retailer_Site["END_DATE"] = END_TIME
DIM_Retailer_Site["ACTIVE_INDICATOR"] = 1
DIM_Retailer_Site["SK_RETAILER_SITE_CODE"] = DIM_Retailer_Site.index + 1
#DIM_Retailer_Site
# customer_store
DIM_Retailer_Site

,RETAILER_SITE_CODE,RETAILER_CODE,ADRESS1,ADRESS2,CITY,REGION,POSTAL_CODE,ACTIVE_INDICATOR,COUNTRY,ORIGIN,CUSTOMER,START_DATE,END_DATE,SK_RETAILER_SITE_CODE
0,1,NaN,1117 Franklin Blvd,-,Winnipeg,Manitoba,R2C 0M5,1,4,customer_store,89.0,2026-05-18,9999-12-31,1
1,1,89.0,1117 Franklin Blvd,-,Winnipeg,Manitoba,R2C 0M5,1,4,retailer_site,NaN,2026-05-18,9999-12-31,2
2,2,NaN,"45, rue Atwater",-,Montréal,Québec,H2T 9K8,1,4,customer_store,89.0,2026-05-18,9999-12-31,3
3,2,89.0,"45, rue Atwater",-,Montréal,Québec,H2T 9K8,1,4,retailer_site,NaN,2026-05-18,9999-12-31,4
4,3,NaN,328 Hodgson Road,-,Fredericton,New Brunswick,E3B 2H2,1,4,customer_store,89.0,2026-05-18,9999-12-31,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,441,NaN,Amselweg 8,-,Innsbruck,California,A-6020,1,5,customer_store,209.0,2026-05-18,9999-12-31,748
748,442,209.0,Hasengasse 38,-,Ischgl,-,A-6561,1,5,retailer_site,NaN,2026-05-18,9999-12-31,749
749,442,NaN,Hasengasse 38,-,Ischgl,California,A-6561,1,5,customer_store,209.0,2026-05-18,9999-12-31,750
750,443,210.0,Hammerstraße 26,-,Innsbruck,-,A-6020,1,5,retailer_site,NaN,2026-05-18,9999-12-31,751


DIM_Customer

In [11]:
customer_type = customer_type.rename(columns={'CUSTOMER_TYPE_CODE' : 'CUSTOMER_TYPE'})
DIM_customer = customer.merge(customer_type, how='left', on=['CUSTOMER_TYPE'])
DIM_customer = DIM_customer.rename(columns={'CUSTOMER_CODEMR':'CUSTOMER_HEADQUARTERS'})

#DIM_customer

DIM_returned_item

In [12]:
returned_item = returned_item.rename(columns={'RETURN_REASON' : 'RETURN_REASON_CODE'})
return_reason = return_reason.rename(columns={'RETURN_DESCRIPTION' : 'RETURN_REASON_EN'})

DIM_returned_item = returned_item.merge(return_reason,on='RETURN_REASON_CODE',how='left')

DIM_returned_item = DIM_returned_item.rename(columns={'ORDER_DETAIL_CODE':'ORDER_DETAILS', 'RETURN_QUANTITY':'QUANTITY_CATEGORY', 'RETURN_DESCRIPTION_EN':'RETURN_REASON_EN'})
DIM_returned_item['RETURN_DATE'] = DIM_returned_item['RETURN_DATE'].astype(str)

DIM_returned_item

,RETURN_CODE,RETURN_DATE,QUANTITY_CATEGORY,ORDER_DETAILS,RETURN_REASON_CODE,RETURN_REASON_EN
0,1491,01-Aug-2024 04:10:24 AM,8,84858,5,Unsatisfactory product
1,1492,06-Dec-2023 06:46:19 PM,2,84440,2,Incomplete product
2,1493,24-Jun-2024 09:23:14 AM,22,84867,4,Wrong product shipped
3,1494,21-Jul-2024 12:00:09 AM,20,84873,3,Wrong product ordered
4,1496,07-Oct-2023 05:13:58 AM,2,84488,1,Defective product
...,...,...,...,...,...,...
685,2465,06-Jun-2024 09:22:12 AM,24,114972,3,Wrong product ordered
686,2466,06-Oct-2024 12:31:25 PM,66,114983,5,Unsatisfactory product
687,2467,31-Mar-2025 03:39:38 PM,64,115215,4,Wrong product shipped
688,2468,23-Dec-2025 07:48:50 PM,2,115171,1,Defective product


DIM_Month

In [13]:
DIM_Month = pd.DataFrame()

DIM_Month = pd.concat([
    pd.DataFrame({
        'YEAR': inventory_levels['INVENTORY_YEAR'],
        'MONTHNR': inventory_levels['INVENTORY_MONTH']
    }),
    
    pd.DataFrame({
        'YEAR': product_forecast['YEAR'],
        'MONTHNR': product_forecast['MONTH']
    })
]).drop_duplicates().sort_values(['YEAR', 'MONTHNR']).reset_index(drop=True)

DIM_Month['MONTH'] = DIM_Month['MONTHNR'].map({1 : 'Jan', 2 : 'Feb', 3 : 'Mar', 4 : 'Apr', 5:'Mei', 6 : 'Jun', 7 : 'Jul', 8 : 'Aug', 9 : 'Sep', 10 : 'Oct', 11 : 'Nov', 12 : 'Dec'})
DIM_Month['QUARTER'] = DIM_Month['MONTHNR'].map({ 1: 1, 2 : 1, 3 : 1, 4 : 2, 5:2, 6 : 2, 7 : 3, 8 : 3, 9 : 3, 10 : 4, 11 : 4, 12 : 4})
DIM_Month['MONTH_ID'] = DIM_Month.index

#DIM_Month

DIM_Datum

In [14]:
DIM_Date = pd.DataFrame()

DIM_Date['DATUM'] = pd.concat([
    pd.to_datetime(returned_item['RETURN_DATE'], format="%d-%b-%Y %I:%M:%S %p"),
    pd.to_datetime(product['INTRODUCTION_DATE'], format="%d-%b-%Y %I:%M:%S %p"),
    pd.to_datetime(order_header['ORDER_DATE'], format="%d-%b-%Y %I:%M:%S %p"),
    pd.to_datetime(sales_staff['DATE_HIRED'], format="%d-%b-%Y %I:%M:%S %p")
]).drop_duplicates().sort_values().reset_index(drop=True)
DIM_Date['DAY'] = pd.to_datetime(DIM_Date['DATUM']).dt.day

DIM_Date['YEAR'] = DIM_Date['DATUM'].dt.year
DIM_Date['MONTHNR'] = DIM_Date['DATUM'].dt.month

DIM_Date = DIM_Date.merge(
    DIM_Month[['YEAR', 'MONTHNR', 'MONTH_ID']],
    on=['YEAR', 'MONTHNR'],
    how='left'
)
DIM_Date = DIM_Date.rename(columns={'MONTHNR' : 'MONTH'})
DIM_Date['DATUM'] = DIM_Date['DATUM'].astype(str)
DIM_Date['DATE_ID'] = DIM_Date.index

#DIM_Date

DIM_Product

In [15]:
product = product.rename(columns={'PRODUCT_TYPE' : 'PRODUCT_TYPE_CODE'})

DIM_product = product.merge(product_type, how='left', on='PRODUCT_TYPE_CODE')
DIM_product = DIM_product.rename(columns={'PRODUCT_LINE' : 'PRODUCT_LINE_CODE'})

DIM_product = DIM_product.merge(product_line, how='left', on='PRODUCT_LINE_CODE')
DIM_product = DIM_product.rename(columns={'PRODUCT_IMAGE':'PRODUCT_IMG', 'PRODUCTION_COST':'COST_CATEGORY'})
DIM_product['INTRODUCTION_DATE'] = DIM_product['INTRODUCTION_DATE'].astype(str)
DIM_product["START_DATE"] = pd.Timestamp.now().strftime("%Y-%m-%d")
DIM_product["END_DATE"] = END_TIME
DIM_product["ACTIVE_INDICATOR"] = 1
#DIM_product

FACT_Order_Header

In [16]:
order_method = order_method.rename(columns={'ORDER_METHOD_CODE' : 'ORDER_METHOD'})

FACT_order_header = order_header.merge(order_method, how='left', on='ORDER_METHOD')
FACT_order_header = FACT_order_header.rename(columns={'SALES_STAFF_CODE':'SALES_STAFF', 'RETAILER_SITE_CODE':'RETAILER_SITE', 'SALES_BRANCH_CODE':'SALES_BRANCH'})
FACT_order_header['ORDER_DATE'] = FACT_order_header['ORDER_DATE'].astype(str)

#FACT_order_header

FACT_Order_Details

In [17]:
FACT_order_details = order_details.copy()

FACT_order_details['UNIT_PRICE'] = FACT_order_details['UNIT_PRICE'].astype('double')
FACT_order_details['UNIT_COST'] = FACT_order_details['UNIT_COST'].astype('double')
FACT_order_details['UNIT_REVENUE'] = (FACT_order_details['UNIT_PRICE'] - FACT_order_details['UNIT_COST'])

#FACT_order_details

DWH inladen

In [18]:

dwh_cursor.execute("PRAGMA foreign_keys = OFF;")

# =========================
# 1. ROOT TABLES
# =========================

root = [
    ("DIM_age_group", age_group, [
        "AGE_GROUP_CODE",
        "UPPER_AGE",
        "LOWER_AGE"
    ]),

    ("DIM_customer_segment", customer_segment, [
        "SEGMENT_CODE",
        "LANGUAGE",
        "SEGMENT_NAME",
        "SEGMENT_DESCRIPTION"
    ]),

    ("DIM_Month", DIM_Month, [
        "MONTH_ID",
        "MONTHNR",
        "MONTH",
        "QUARTER",
        "YEAR"
    ]),
]

# =========================
# 2. MID LAYER
# =========================

mid = [

    ("DIM_country", DIM_Country, [
        "COUNTRY_CODE",
        "COUNTRY",
        "LANGUAGE",
        "CURRENCY",
        "FLAG_IMAGE",
        "TERRITORY_NAME_EN"
    ]),

    ("DIM_Date", DIM_Date, [
        "DATE_ID",
        "DATUM",
        "MONTH"
    ]),

    ("DIM_sales_branch", DIM_Sales_Branch, [
        "SALES_BRANCH_CODE",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_CODE",
        "COUNTRY"
    ]),

    ("DIM_product", DIM_product, [
        "PRODUCT_NUMBER",
        "DESCRIPTION",
        "INTRODUCTION_DATE",
        "COST_CATEGORY",
        "MARGIN",
        "PRODUCT_IMG",
        "LANGUAGE",
        "PRODUCT_NAME",
        "PRODUCT_TYPE_EN",
        "PRODUCT_LINE_EN",
        "START_DATE",
        "END_DATE",
        "ACTIVE_INDICATOR"
    ]),
]

# =========================
# 3. THIRD LAYER
# =========================

third = [

    ("DIM_customer_headquarters", customer_headquarters, [
        "CUSTOMER_CODE_HQ",
        "CUSTOMER_NAME",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_ZONE",
        "PHONE",
        "FAX",
        "COUNTRY",
        "SEGMENT_CODE"
    ]),

    ("DIM_sales_staff", DIM_sales_Staff, [
        "SALES_STAFF_CODE",
        "FIRST_NAME",
        "LAST_NAME",
        "POSITION_EN",
        "WORK_PHONE",
        "EXTENSION",
        "FAX",
        "DATE_HIRED",
        "TRAINING_YEAR",
        "TRAINING_COURSE",
        "STATISFACTION_YEAR",
        "STATISFACTION_DESCRIPTION",
        "SALES_BRANCH",
        "MANAGER"
    ]),

    ("DIM_customer", DIM_customer, [
        "CUSTOMER_CODE",
        "COMPANY_NAME",
        "CUSTOMER_TYPE_EN",
        "CUSTOMER_HEADQUARTERS"
    ]),

    ("DIM_retailer_site", DIM_Retailer_Site, [
        "SK_RETAILER_SITE_CODE",
        "RETAILER_SITE_CODE",
        "RETAILER_CODE",
        "ADRESS1",
        "ADRESS2",
        "CITY",
        "REGION",
        "POSTAL_CODE",
        "START_DATE",
        "END_DATE",
        "COUNTRY",
        "CUSTOMER",
        "ACTIVE_INDICATOR",
        "ORIGIN"
    ]),
]

# =========================
# 4. FACT TABLES
# =========================

fact = [

    ("FACT_sales_demographic", sales_demographic, [
        "DEMOGRAPHIC_CODE",
        "SALES_PERCENT",
        "CUSTOMER_HEADQUARTERS",
        "AGE_GROUP"
    ]),

    ("FACT_order_header", FACT_order_header, [
        "ORDER_NUMBER",
        "RETAILER_NAME",
        "RETAILER_CONTACT_CODE",
        "ORDER_DATE",
        "ORDER_METHOD_EN",
        "RETAILER_SITE",
        "SALES_STAFF",
        "SALES_BRANCH"
    ]),

    ("FACT_order_details", FACT_order_details, [
        "ORDER_DETAIL_CODE",
        "QUANTITY",
        "UNIT_COST",
        "UNIT_REVENUE",
        "UNIT_PRICE",
        "UNIT_SALE_PRICE",
        "ORDER_HEADER",
        "PRODUCT"
    ]),

    ("DIM_returned_item", DIM_returned_item, [
        "RETURN_CODE",
        "RETURN_DATE",
        "QUANTITY_CATEGORY",
        "RETURN_REASON_EN",
        "ORDER_DETAILS"
    ]),

    ("DIM_inventory_level", inventory_levels, [
        "INVENTORY_LEVELS_CODE",
        "INVENTORY_YEAR",
        "INVENTORY_MONTH",
        "INVENTORY_COUNT",
        "PRODUCT"
    ]),

    ("DIM_product_forecast", product_forecast, [
        "PRODUCT_FORECAST_CODE",
        "YEAR",
        "MONTH",
        "EXPECTED_VOLUME",
        "PRODUCT"
    ]),

    ("FACT_sales_target", sales_target, [
        "SALES_TARGET_CODE",
        "SALES_YEAR",
        "SALES_PERIOD",
        "RETAILER_NAME",
        "RETAILER_CODE",
        "SALES_TARGET",
        "SALES_STAFF",
        "PRODUCT"
    ]),
]
# =========================
# ALL TABLES
# =========================

all_tables = root + [table for table in mid] + third + fact

for table_name, df, cols in all_tables:

    missing = set(cols) - set(df.columns)

    if missing:
        print(f"❌ {table_name} mist kolommen: {missing}")
        logger.error(f"DWH|ERROR|Data/sdm-dwh/dwhdb.db|{table_name}|MISSING COLUMNS: {str(missing)}")
        continue

    df_clean = df[cols]

    query = f"""
    INSERT OR REPLACE INTO {table_name}
    ({",".join(cols)})
    VALUES ({",".join(["?"] * len(cols))})
    """

    try:
        dwh_cursor.executemany(query, df_clean.values.tolist())
        print(f"✔ {table_name}")
        logger.info(f"DWH|INFO|Data/sdm-dwh/dwhdb.db|{table_name}|{str("SUCCESS")}")

    except Exception as e:
        print(f"❌ {table_name} ERROR: {e}")
        logger.error(f"DWH|ERROR|Data/sdm-dwh/dwhdb.db|{table_name}|{str(e)}")
        break

dwh_conn.commit()

✔ DIM_age_group
✔ DIM_customer_segment
✔ DIM_Month
✔ DIM_country
✔ DIM_Date
✔ DIM_sales_branch
✔ DIM_product
✔ DIM_customer_headquarters
✔ DIM_sales_staff
✔ DIM_customer
✔ DIM_retailer_site
✔ FACT_sales_demographic
✔ FACT_order_header
✔ FACT_order_details
✔ DIM_returned_item
✔ DIM_inventory_level
✔ DIM_product_forecast
✔ FACT_sales_target


DWH resetten

In [19]:
def resetDWH():
    """
    Verwijdert alle tabellen uit het DWH
    + logging
    """

    try:
        logger.info("DWH|RESET_DWH_START|Data/dwhdb.db.db|||||START")

        dwh_cursor.execute(
            "PRAGMA foreign_keys = OFF;"
        )

        tables = dwh_cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type='table'
        """).fetchall()

        if not tables:
            logger.warning("DWH|ERROR|Data/dwhdb.db||||FAILED|%s",str(e))

        for (table,) in tables:

            try:
                dwh_cursor.execute(
                    f"DROP TABLE IF EXISTS {table}"
                )

                logger.info(
                "DWH|READ|Data/dwhdb.db|%s|||SUCCESS",
                table
                )

                print(f"Dropped: {table}")

            except Exception as table_error:

                logger.error(
                f"DWH|ERROR|Data/dwhdb.db.db|%s|||FAILED|%s",
                table,
                str(e)
                )

        dwh_conn.commit()

        logger.info("DWH|LOAD_DWH_END|Data/dwhdb.db|||||END")
        print("DWH volledig gereset (DROP)")

    except Exception as e:

        dwh_conn.rollback()

        logger.error(
            f"logger.error("
            "DWH|ERROR|Data/dwhdb.db||||FAILED|%s",
            str(e)
            )

        print(f"ERROR: {e}")

    finally:

        dwh_conn.execute(
            "PRAGMA foreign_keys = ON;"
        )

        logger.info("DWH|RESET_DWH_END|%s|||||END")


# RUN
# resetDWH()